In [1]:
import pandas as pd

In [2]:
df1 = pd.read_csv('Data/top_40_cities_data.csv')
cities = df1['Name'].unique().tolist()

missing_values_summary = []
skip_cities = ["Banning Airport", "El Rio-Rio Mesa School #2", "Hollister", "San Rafael", "Santa Cruz", "Modesto-14th Street", "Lake Elsinore", "Laney College", "Madera-City", "Mesa2", "Piru - Pacific", "Reseda", "Santa Clarita"]

for city in cities:
    
    if city in skip_cities:
        print(f"Skipping city: {city}")
        continue 
    print(f"Processing city: {city}")    
    df1 = pd.read_csv('Data/top_40_cities_data.csv')
    
    
    df1['Date'] = pd.to_datetime(df1['Date'])
    df1 = df1.sort_values(['Date'])
    df1 = df1[df1['Name'] == city]
    df1 = df1.reset_index(drop=True)

    df1 = df1.rename(columns={
        'Latitude': 'latitude',
        'Longitude': 'longitude',
        'PM2.5': 'pm2.5',
        'Ozone': 'ozone',
        'AQI': 'aqi',
        'Name': 'city',
        'Date': 'date'
    })
    df1.set_index(['date'], inplace=True)

    weather_file = f'Data/Weather/historical_weather_data_{city.replace(" ", "_")}.csv'
    df2 = pd.read_csv(weather_file)
    df2['date'] = pd.to_datetime(df2['date'])
    df2.set_index('date', inplace=True)

    df2 = (
        df2.groupby('City')
        .resample('D')
        .mean()
        .reset_index()
    )
    df2.set_index(['date'], inplace=True)
    df2.drop(['City'], inplace=True, axis=1)

    # Merge pollution and weather data
    df = pd.merge(df1, df2, left_index=True, right_index=True, how='inner')

    df.reset_index(inplace=True)

    # Create full date range and merge
    date_range = pd.date_range(start='2014-01-01', end='2025-04-04', freq='D')
    full_dates_df = pd.DataFrame(date_range, columns=['date'])
    df = pd.merge(full_dates_df, df, on='date', how='left')

    # Forward-fill and backward-fill missing data
    missing_count = df.isnull().sum().sum()/11  # Total missing values before filling
    df = df.ffill().bfill()

    output_file = f'Data/Final/{city.replace(" ", "_").replace("-", "_")}.csv'
    df.to_csv(output_file, index=False)

    missing_values_summary.append({'city': city, 'missing_values': missing_count})

missing_values_df = pd.DataFrame(missing_values_summary)

Processing city: Bakersfield-California
Skipping city: Banning Airport
Processing city: Calexico-Ethel Street
Processing city: Carmel Valley
Skipping city: El Rio-Rio Mesa School #2
Processing city: Fresno - Garland
Skipping city: Hollister
Processing city: King City 2
Skipping city: Modesto-14th Street
Processing city: Oakland
Processing city: Ojai - East Ojai Ave
Skipping city: Piru - Pacific
Processing city: Salinas 3
Processing city: San Jose - Jackson
Skipping city: San Rafael
Skipping city: Santa Clarita
Skipping city: Santa Cruz
Processing city: Simi Valley-Cochran Street
Processing city: Temecula
Processing city: Thousand Oaks


In [3]:
print("Missing Values Summary:")
missing_values_df

Missing Values Summary:


,city,missing_values
0,Bakersfield-California,149.454545
1,Calexico-Ethel Street,177.818182
2,Carmel Valley,186.545455
3,Fresno - Garland,181.090909
4,King City 2,178.909091
5,Oakland,176.727273
6,Ojai - East Ojai Ave,175.636364
7,Salinas 3,116.727273
8,San Jose - Jackson,229.090909
9,Simi Valley-Cochran Street,123.272727
